# SC2001 Project 2 - Part (c): Comparing the Two Implementations

**Task.** Compare the two implementations of Dijkstra's algorithm from parts (a) and (b), and
discuss which one is better and in what circumstances.

| | Part (a) | Part (b) |
|---|---|---|
| Graph representation | adjacency **matrix** | array of adjacency **lists** |
| Priority queue | **array** (linear scan for the minimum) | **minimizing heap** (`heapq`) |

**This notebook builds on parts (a) and (b) and does not repeat them.** It reuses, unchanged:
- from the Part (a) notebook: `INF`, `SEED`, `MAX_WEIGHT`, `dijkstra_matrix_array`,
  `random_graph_edges`, `to_matrix`, `bellman_ford`, `log_slope`
- from Part (b) (`project2_part_b.py`): `dijkstraB`

Run the Part (a) cells before this notebook. Everything defined below will be new to Part (c).

**Contents**
1. Theoretical comparison
2. Tools used only by Part (c)
3. Check that both implementations agree
4. Empirical comparison
   - 4.1 vary $|V|$
   - 4.2 vary $|E|$ and find the crossover
   - 4.3 a cost model for the crossover
   - 4.4 a worst-case input for (b)
   - 4.5 operation counts: what the heap really does
5. Discussion: which is better, and when
6. Conclusion

## Setup

In [ ]:
%matplotlib inline

import gc
import heapq
import math
import random
import time

import matplotlib.pyplot as plt

QUICK_C = False    # True = small sizes for a fast run; False = full run (a few minutes)

# Part (b) lives in project2_part_b.py. Import it unless it is already defined in this notebook.
if "dijkstraB" not in globals():
    from project2_part_b import dijkstraB

# Everything else is reused from the Part (a) notebook, which must be run first.
REQUIRED_FROM_A = ["INF", "SEED", "MAX_WEIGHT", "dijkstra_matrix_array", "random_graph_edges",
                   "to_matrix", "bellman_ford", "log_slope"]
missing = [name for name in REQUIRED_FROM_A if name not in globals()]
if missing:
    raise NameError(f"Run the Part (a) cells first. Missing names: {missing}")

---
## 1. Theoretical comparison

Throughout, $V = |V|$ and $E = |E|$, and the graph is simple and directed, so $E < V^2$.

### 1.1 What we take from parts (a) and (b)

- **(a)** reads exactly $2V^2$ cells (a full array scan and a full matrix row per vertex), so it runs
  in $\Theta(V^2)$ in every case and does not depend on $E$ or on the weights.
- **(b)** scans each adjacency list once ($\Theta(V+E)$ in total). Because `dijkstraB` has no
  decrease-key, every successful relaxation pushes a new heap entry and stale entries are skipped
  when popped (*lazy deletion*). There are therefore at most $E+1$ pushes, each costing $O(\log E) = O(\log V)$,
  so $T_b = O\big((V+E)\log V\big)$ in the worst case. With only about $V$ pushes the cost drops to
  $O(E + V\log V)$, so **the running time of (b) also depends on the weights**.

### 1.2 Side by side

| | (a) matrix + array | (b) adjacency lists + heap |
|---|---|---|
| Time | $\Theta(V^2)$ in every case | $O((V+E)\log V)$ worst case; $O(E + V\log V)$ if pushes are rare |
| Space for the graph | $\Theta(V^2)$ | $\Theta(V + E)$ |
| Depends on $E$? | No | Yes |
| Depends on the weights? | No | Yes (through the number of heap pushes) |

### 1.3 Which one is faster?

| Graph | (a) | (b) | Better |
|---|---|---|---|
| Sparse, $E = \Theta(V)$ | $\Theta(V^2)$ | $\Theta(V\log V)$ | **(b)**, by a factor of about $V/\log V$ |
| In between, $E = \Theta(V^{1.5})$ | $\Theta(V^2)$ | at most $\Theta(V^{1.5}\log V)$ | **(b)** |
| Dense, $E = \Theta(V^2)$ | $\Theta(V^2)$ | $\Theta(V^2)$ to $\Theta(V^2\log V)$ | **(a)** in the worst case, otherwise the same order |

Setting the worst case of (b) equal to (a), with constants $c_a$ and $c_b$ for the cost of one
basic step in each implementation:

$$
c_a V^2 \approx c_b\,E\log V
\quad\Longrightarrow\quad
E^* \approx \frac{c_a}{c_b}\cdot\frac{V^2}{\log V}.
$$

Below $E^*$ the heap version wins; above it the matrix version wins. The theory therefore predicts
the *shape* of the comparison. The position of the crossover depends on $c_a$ and $c_b$, which
depend on the language and the machine, so it has to be measured.

### 1.4 Space

(a) always stores $V^2$ cells, even for a graph with no edges. In CPython this is 8 bytes per
cell for the pointer alone, so $V = 10^4$ already needs about 0.8 GB, and $V = 10^5$ is out of
reach. (b) stores only $V + E$ entries.

### 1.5 Predictions to test empirically
1. **Sparse graphs** ($E = 5V$): (b) is much faster than (a), and the speed-up grows with $V$.
2. **Fixed $V$, increasing $E$**: the time of (a) stays flat, the time of (b) grows with $E$, and the
   two curves cross at some density.
3. **Worst case for (b)**: on a graph where every edge causes a heap push, (b) should lose to (a)
   by a growing margin.
4. **Operation counts**: (a) always reads exactly $2V^2$ cells. (b) scans $E$ edges and, depending
   on the weights, does between about $V$ and $E$ heap pushes.

---
## 2. Tools used only by Part (c)

The implementations themselves come from parts (a) and (b). Four small additions are needed here:

- `to_adj_list` builds the adjacency-list form that `dijkstraB` expects. Each random graph is built
  once as a matrix (with `to_matrix` from Part (a)) and once as adjacency lists, so both
  implementations always run on **the same graph**. Building the graphs is not timed.
- `dijkstra_matrix_array_lean` is Part (a)'s function with its three operation counters removed. The
  counters add work inside both inner loops, which would slow (a) down and unfairly favour (b) in the
  timings. The logic is identical; the next section checks that the answers are the same.
- `dijkstraB_counted` is Part (b)'s function with counters added (edge scans, heap pushes,
  stale pops, heap size). It is used only in 4.5.
- `adversarial_edges(n)` builds a graph on which **every edge triggers a heap push**, the worst
  case for (b). It is explained in 4.4.

The random graphs come from `random_graph_edges` in Part (a): exactly $m$ distinct directed edges,
random integer weights in $[1, 100]$, and a random spanning tree from vertex 0 so that every vertex is
reachable. Random weights are the *typical* case for (b); `adversarial_edges` gives its *worst* case.

In [ ]:
def to_adj_list(n, edges):
    # The array-of-adjacency-lists form that Part (b)'s dijkstraB expects:
    # graph[u] is a list of (neighbour, weight) pairs.
    graph = [[] for _ in range(n)]
    for u, v, w in edges:
        graph[u].append((v, w))
    return graph


def adversarial_edges(n):
    # A graph built to be the WORST case for the heap version (b).
    # Chain edges i -> i+1 have weight 1, so the true distance of vertex v is v.
    # Every other forward edge i -> j (j >= i+2) has weight 2(j-i)+1. When vertex i is
    # processed it offers j the distance 2j-i+1, which is always one less than what
    # vertex i-1 offered. So EVERY edge succeeds in relaxing and triggers a heap push.
    edges = [(i, i + 1, 1) for i in range(n - 1)]
    for i in range(n):
        for j in range(i + 2, n):
            edges.append((i, j, 2 * (j - i) + 1))
    return edges


def dijkstra_matrix_array_lean(graph, source):
    # Part (a)'s algorithm with its operation counters removed. The logic is unchanged.
    # Part (a)'s version updates three counters inside both inner loops, which would slow
    # (a) down and make the timing comparison with (b) unfair, so timings use this copy.
    n = len(graph)
    d = [INF] * n
    pi = [None] * n
    S = [False] * n
    d[source] = 0
    for _ in range(n):
        u = -1
        best = INF
        for v in range(n):
            if not S[v] and d[v] < best:
                best = d[v]
                u = v
        if u == -1:
            break
        S[u] = True
        row = graph[u]
        for v in range(n):
            w = row[v]
            if w != INF and not S[v] and d[u] + w < d[v]:
                d[v] = d[u] + w
                pi[v] = u
    return d, pi


def dijkstraB_counted(graph, source):
    # Part (b)'s dijkstraB with counters added. Used only for the counts in 4.5.
    distances = [INF] * len(graph)
    distances[source] = 0
    min_heap = [(0, source)]
    ops = {"edge_scans": 0, "pushes": 1, "pops": 0, "stale_pops": 0, "max_heap": 1}

    while min_heap:
        current_distance, vertex = heapq.heappop(min_heap)
        ops["pops"] += 1
        if current_distance > distances[vertex]:
            ops["stale_pops"] += 1
            continue
        for neighbour, weight in graph[vertex]:
            ops["edge_scans"] += 1
            new_distance = current_distance + weight
            if new_distance < distances[neighbour]:
                distances[neighbour] = new_distance
                heapq.heappush(min_heap, (new_distance, neighbour))
                ops["pushes"] += 1
                if len(min_heap) > ops["max_heap"]:
                    ops["max_heap"] = len(min_heap)
    return distances, ops

---
## 3. Check that both implementations agree

Parts (a) and (b) test their own code. Before comparing speed we check that the versions used
here give identical answers on the same graphs, including the extra copies added in section 2:

1. 200 random graphs, checked against Bellman-Ford (`bellman_ford` from Part (a))
2. A graph with an unreachable vertex, whose distance must stay $\infty$
3. The adversarial graphs, where the true distance of vertex $v$ is $v$

In [ ]:
def run_all(n, edges, source=0):
    # (a) as written in Part (a), (a) without counters, (b) as written in Part (b), (b) with counters
    dA, _, _ = dijkstra_matrix_array(to_matrix(n, edges), source)
    dL, _ = dijkstra_matrix_array_lean(to_matrix(n, edges), source)
    dB = dijkstraB(to_adj_list(n, edges), source)
    dC, _ = dijkstraB_counted(to_adj_list(n, edges), source)
    return dA, dL, dB, dC


# 1. Random graphs: every version agrees with Bellman-Ford
rng = random.Random(SEED)
for _ in range(200):
    n = rng.randint(2, 40)
    m = rng.randint(n - 1, n * (n - 1))
    edges = random_graph_edges(n, m, rng)
    ref = bellman_ford(n, edges, 0)
    assert all(d == ref for d in run_all(n, edges))

# 2. A graph with an unreachable vertex
assert all(d == [0, 4, INF] for d in run_all(3, [(0, 1, 4)]))

# 3. The adversarial graph: the true distance of vertex v is v
for n in (5, 30, 200):
    assert all(d == list(range(n)) for d in run_all(n, adversarial_edges(n)))

print("Both implementations (and their instrumented / lean copies) agree with each other and")
print("with Bellman-Ford on 200 random graphs, an unreachable vertex and the adversarial graphs.")

---
## 4. Empirical comparison

`compare(n, edges, reps)` runs both algorithms on the same graph and returns the fastest time of
several runs for each, with the garbage collector paused (the same method as Part (a)).
`log_slope` (from Part (a)) fits $\log(\text{time}) = k\log V + c$, so the slope $k$ is the measured
exponent in $\text{time}\propto V^k$.

**All timings are from pure Python on one machine.** The absolute times, and the position of the
crossover in 4.2, will differ on another machine. The shapes of the curves should not. Timings also
vary a little from run to run, so the observations below quote ranges rather than the exact figures
printed once.

In [ ]:
def time_it(func, graph, reps, min_total=0.3, max_runs=60):
    # Fastest of several runs, garbage collector paused (same idea as Part (a)).
    # Very short runs are repeated until 0.3 s of total run time has been spent, so that
    # timer noise does not distort them.
    best, total, runs = INF, 0.0, 0
    gc.disable()
    try:
        while runs < reps or (total < min_total and runs < max_runs):
            start = time.perf_counter()
            func(graph, 0)
            dt = time.perf_counter() - start
            best = min(best, dt)
            total += dt
            runs += 1
    finally:
        gc.enable()
    return best


def compare(n, edges, reps):
    # Build both representations of the SAME graph (not timed), then time each algorithm.
    matrix = to_matrix(n, edges)
    adj = to_adj_list(n, edges)
    return time_it(dijkstra_matrix_array_lean, matrix, reps), time_it(dijkstraB, adj, reps)


def crossover(rows):
    # Density where time(a)/time(b) first drops below 1, by linear interpolation in log-ratio.
    # rows = list of (density, |E|, time_a, time_b), sorted by density.
    for (d0, _, a0, b0), (d1, _, a1, b1) in zip(rows, rows[1:]):
        r0, r1 = math.log(a0 / b0), math.log(a1 / b1)
        if r0 > 0 >= r1:
            return d0 + (d1 - d0) * r0 / (r0 - r1)
    return None


if QUICK_C:
    C_SIZES, C_REPS = list(range(100, 801, 100)), 2
    C_CROSS_V = (300, 600)
    C_ADV_SIZES = list(range(100, 501, 100))
else:
    C_SIZES, C_REPS = list(range(200, 2001, 200)), 3
    C_CROSS_V = (500, 1000, 1500)
    C_ADV_SIZES = list(range(200, 1201, 200))

### 4.1 Varying $|V|$

$V$ runs from 200 to 2000 at two densities, the same two as in Part (a):
- **sparse**: $E = 5V$
- **dense**: $E = V(V-1)/2$, half of all possible edges

Theory predicts: sparse $\to$ (a) $\Theta(V^2)$ against (b) about $\Theta(V\log V)$; dense $\to$ both about $V^2$.

In [ ]:
rng = random.Random(SEED)
sweep_v = {"sparse": [], "dense": []}

header = (f"{'|V|':>6} {'graph':>7} {'|E|':>9} {'time (a) s':>11} {'time (b) s':>11} "
          f"{'(a)/(b)':>8}")
print(header)
print("-" * len(header))
for n in C_SIZES:
    for label, m in (("sparse", 5 * n), ("dense", n * (n - 1) // 2)):
        tA, tB = compare(n, random_graph_edges(n, m, rng), C_REPS)
        sweep_v[label].append((n, m, tA, tB))
        print(f"{n:>6} {label:>7} {m:>9} {tA:>11.4f} {tB:>11.4f} {tA / tB:>8.2f}")

print()
for label, rows in sweep_v.items():
    ns = [r[0] for r in rows]
    kA = log_slope(ns, [r[2] for r in rows])
    kB = log_slope(ns, [r[3] for r in rows])
    print(f"{label:>6}: fitted exponent  (a) k = {kA:.2f}   (b) k = {kB:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for ax, label in zip(axes[:2], ("sparse", "dense")):
    rows = sweep_v[label]
    ax.plot([r[0] for r in rows], [r[2] for r in rows], "o-", label="(a) matrix + array")
    ax.plot([r[0] for r in rows], [r[3] for r in rows], "s-", label="(b) adj. lists + heap")
    desc = "|E| = 5|V|" if label == "sparse" else "|E| = |V|(|V|-1)/2"
    ax.set(title=f"Running time vs |V|, {label} graph ({desc})", xlabel="|V|", ylabel="time (s)")
    ax.legend()
    ax.grid(alpha=0.3)

ax = axes[2]
for label, marker in (("sparse", "o"), ("dense", "s")):
    rows = sweep_v[label]
    ax.plot([r[0] for r in rows], [r[2] / r[3] for r in rows], marker + "-", label=f"{label} graph")
ax.axhline(1, ls=":", color="black")
ax.set_yscale("log")
ax.set(title="Speed-up of (b) over (a):  time(a) / time(b)", xlabel="|V|",
       ylabel="ratio, log scale  (>1 means (b) is faster)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

**Observations**
- **Sparse graphs: (b) wins by a large and growing margin.** (a) grows with exponent
  $k \approx 2.1$-$2.2$, matching $\Theta(V^2)$. (b) grows with $k \approx 1.1$-$1.3$, close to the
  $V\log V$ prediction. The speed-up goes from about 15x at $V = 200$ to over 100x at the
  largest sizes (about 75x to 135x for $V \ge 1000$). This confirms prediction 1.
- **Half-dense graphs: the two are close.** The speed-up of (b) is only about 3x at $V = 200$ and
  settles within roughly 1.5x of parity from $V \approx 600$ on. Which side wins depends on the
  machine and the run (in our figures (b) stays about 1.3x ahead). Half-dense is close to the crossover,
  which is exactly what 4.2 shows.
- **(b) can grow faster than $V^2$ on the dense graphs.** The extra $\log V$ factor would give an exponent
  of only about 2.2, and the heap does very little work on these graphs (see 4.5), so the per-edge
  cost of (b) rises with $V$. 4.3 measures this and gives a likely reason.

### 4.2 Varying $|E|$ and finding the crossover

For three fixed values of $V$, the density $E / (V(V-1))$ goes from a tree ($E = V-1$, about 0.2%)
up to a complete directed graph (100%). Theory predicts that (a) stays flat, (b) grows with $E$, and
that they cross. `crossover` estimates the density at which $\text{time}(a)/\text{time}(b)$ drops
below 1, by interpolation.

In [ ]:
FRACTIONS = [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
rng = random.Random(SEED + 1)
sweep_e = {}      # |V| -> list of (density, |E|, time_a, time_b)

for n in C_CROSS_V:
    max_m = n * (n - 1)
    ms = sorted({max(n - 1, round(max_m * f)) for f in FRACTIONS})
    rows = []
    print(f"|V| = {n}")
    header = f"{'|E|':>9} {'density':>8} {'time (a) s':>11} {'time (b) s':>11} {'(a)/(b)':>8}"
    print(header)
    print("-" * len(header))
    for m in ms:
        tA, tB = compare(n, random_graph_edges(n, m, rng), C_REPS)
        rows.append((m / max_m, m, tA, tB))
        print(f"{m:>9} {m / max_m:>8.1%} {tA:>11.4f} {tB:>11.4f} {tA / tB:>8.2f}")
    sweep_e[n] = rows
    print()

for n, rows in sweep_e.items():
    c = crossover(rows)
    if c is None:
        print(f"|V| = {n}: (b) stays faster than (a) at every density tested")
    else:
        print(f"|V| = {n}: (a) overtakes (b) at density about {c:.0%}  (|E| about {c * n * (n - 1):.0f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

n0 = C_CROSS_V[len(C_CROSS_V) // 2]
rows = sweep_e[n0]
ax = axes[0]
ax.plot([r[0] for r in rows], [r[2] for r in rows], "o-", label="(a) matrix + array")
ax.plot([r[0] for r in rows], [r[3] for r in rows], "s-", label="(b) adj. lists + heap")
ax.set(title=f"Running time vs density  (|V| = {n0})", xlabel="density  |E| / (|V|(|V|-1))", ylabel="time (s)")
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for n, rows in sweep_e.items():
    ax.plot([r[0] for r in rows], [r[2] / r[3] for r in rows], "o-", label=f"|V| = {n}")
ax.axhline(1, ls=":", color="black")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(title="time(a) / time(b) vs density  (below the dotted line, (a) wins)",
       xlabel="density (log scale)", ylabel="ratio (log scale)")
ax.legend()
ax.grid(alpha=0.3, which="both")

fig.tight_layout()
plt.show()

**Observations**
- **(a) is flat; (b) grows with $E$.** Over a range of $E$ spanning more than two orders of
  magnitude, the time of (a) changes only slightly (the small rise comes from the per-cell
  constant, as noted in Part (a)). The time of (b) grows roughly in proportion to $E$.
  This confirms prediction 2.
- **Very sparse: (b) wins by about 75x to 140x** (at 0.2% density). On a tree, (a) still reads
  $2V^2$ cells while (b) does almost no work.
- **Very dense: (a) wins.** On a complete graph, (a) is about 1.4x to 3x faster at $V = 1000$ and
  $V = 1500$. At $V = 500$ the two are roughly level, and in some runs (b) stays ahead even there.
- **The crossover sits at a high density, and the exact value is machine-dependent.** For
  $V \ge 1000$ we saw it at about 60-70% on the machine used for our figures, and at about 25-50% in
  other runs; at $V = 500$ it is poorly defined, because the two curves are nearly equal over a wide
  range of densities. Either way, (b) is ahead over most of the density range, and (a) only wins
  on very dense graphs.

### 4.3 A cost model for the crossover

Section 4.5 will show that on random graphs the heap in (b) does very little work (only a few
thousand pushes, against hundreds of thousands of edges). So for random weights the time of (b) is
essentially "one cost per edge scan", and the time of (a) is "one cost per cell read":

$$
t_a \approx c_A\cdot 2V^2,
\qquad
t_b \approx c_B\cdot E
\quad\Longrightarrow\quad
E^* = 2V^2\,\frac{c_A}{c_B},
\quad\text{i.e. crossover density} \approx 2\,\frac{c_A}{c_B}.
$$

We estimate $c_A$ and $c_B$ from the complete-graph run of each $V$ and compare the predicted
crossover with the one measured in 4.2.

In [ ]:
# Calibrate a simple cost model on the complete-graph run of each |V|, then use it to
# predict where the two curves cross.
#   (a) reads 2|V|^2 cells, each costing about cA seconds           -> time(a) = cA * 2|V|^2
#   (b) on a random graph does about |E| edge scans (heap work is tiny, see 4.5),
#       each costing about cB seconds                                -> time(b) = cB * |E|
# Setting the two equal gives the crossover |E| = 2|V|^2 * cA / cB.
header = (f"{'|V|':>6} {'cA (ns/cell)':>13} {'cB (ns/edge)':>13} {'cB / cA':>8} "
          f"{'predicted':>10} {'measured':>9}")
print(header)
print("-" * len(header))
for n, rows in sweep_e.items():
    density, m, tA, tB = rows[-1]                 # the complete graph
    cA = tA / (2 * n * n)
    cB = tB / m
    predicted = 2 * cA / cB * n / (n - 1)         # as a fraction of |V|(|V|-1)
    measured = crossover(rows)
    ms = f"{measured:>9.0%}" if measured is not None else f"{'none':>9}"
    print(f"{n:>6} {cA * 1e9:>13.1f} {cB * 1e9:>13.1f} {cB / cA:>8.1f} {predicted:>10.0%} {ms}")

**Observations**
- **One number explains the crossover.** A cell read in (a) costs a few tens of nanoseconds,
  because it is a list lookup and a comparison. An edge scan in (b) costs a few times more
  ($c_B/c_A$ between about 2 and 6 in our runs), because it unpacks a tuple, does an addition, a
  comparison and several list accesses. With that ratio the predicted crossover is close to the
  measured one (within about 10 percentage points for $V = 1000$ and $V = 1500$). For $V = 500$ the match is
  looser, because the two curves are nearly parallel over a wide range of densities and the crossover
  is not sharply defined there (when the model predicts a crossover above 100%, (b) did stay ahead at
  every density). So on random graphs the crossover sits at a **fixed fraction of $V^2$**, not at
  $V^2/\log V$ as the worst-case bound would suggest, because the $\log V$ factor of the heap is
  barely paid.
- **The ratio $c_B/c_A$ is larger for larger $V$.** This is why the crossover density falls as $V$
  grows. We have not isolated the cause. A plausible one is memory access: the matrix is a few
  contiguous lists whose cells all point to the same few small integers and to `inf`, whereas the
  adjacency lists hold millions of separate tuples scattered through memory, which makes poor use of
  the CPU cache. This is a property of CPython, and the size of the effect would differ in other
  languages.

### 4.4 A worst-case input for (b)

Random weights are kind to (b). The analysis in 1.1 says its worst case happens when **every**
edge causes a heap push. `adversarial_edges(n)` builds such a graph:
- Chain edges $i \to i+1$ have weight 1, so the true distance of vertex $v$ is $v$.
- Every other forward edge $i \to j$ ($j \ge i+2$) has weight $2(j-i)+1$. When vertex $i$ is
  processed it offers vertex $j$ the distance $2j - i + 1$, which is one less than what vertex
  $i-1$ offered. So every one of the $\approx V^2/2$ edges improves a distance and causes a push.

The graph has the same density as the "half-dense" random graph (about 50%), so the only
difference from 4.1 is the weights. (a) always reads $2V^2$ cells, so the weights can change its time only by a constant factor.

In [ ]:
adv_rows = []
header = (f"{'|V|':>6} {'|E|':>9} {'time (a) s':>11} {'time (b) s':>11} {'(a)/(b)':>8} "
          f"{'heap pushes':>12}")
print(header)
print("-" * len(header))
for n in C_ADV_SIZES:
    edges = adversarial_edges(n)
    tA, tB = compare(n, edges, C_REPS)
    _, ops = dijkstraB_counted(to_adj_list(n, edges), 0)
    adv_rows.append((n, len(edges), tA, tB, ops["pushes"]))
    print(f"{n:>6} {len(edges):>9} {tA:>11.4f} {tB:>11.4f} {tA / tB:>8.2f} {ops['pushes']:>12}")

ns = [r[0] for r in adv_rows]
print(f"\nfitted exponent  (a) k = {log_slope(ns, [r[2] for r in adv_rows]):.2f}   "
      f"(b) k = {log_slope(ns, [r[3] for r in adv_rows]):.2f}")
print("|E| log|V| alone would give an exponent of only about 2.2 here, so part of the growth of (b)")
print("is probably a memory effect: its heap holds hundreds of thousands of tuples (see 4.5).")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
ax.plot(ns, [r[2] for r in adv_rows], "o-", label="(a) matrix + array")
ax.plot(ns, [r[3] for r in adv_rows], "s-", label="(b) adj. lists + heap")
ax.set(title="Adversarial graph: running time vs |V|", xlabel="|V|", ylabel="time (s)")
ax.legend()
ax.grid(alpha=0.3)
ax = axes[1]
ax.plot(ns, [r[4] for r in adv_rows], "o-", label="heap pushes in (b)")
ax.plot(ns, [r[1] for r in adv_rows], ":", color="black", label="|E|")
ax.set(title="Adversarial graph: heap pushes vs |E|", xlabel="|V|", ylabel="count")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**Observations**
- **On this input (b) loses badly.** It is about 3x to 4x slower than (a) at $V = 200$ and about 7x
  slower at $V = 1200$ in our figures (up to about 17x in other runs), and the gap keeps widening.
  The number of heap pushes equals $E+1$ exactly, which is the worst case of section 1.1. This
  confirms prediction 3.
- **The same graph density, with different weights, flips the result.** The half-dense random graph
  of 4.1 is roughly a tie between (a) and (b). With adversarial weights, (b) is several times
  slower. So the comparison depends on the **weights**, not only on $V$ and $E$.
- **(a) changes only by a constant factor.** It reads the same $2V^2$ cells, and its time stays within
  about 1.5x of what it was on the random half-dense graph (more of its relaxations succeed here).
  Its growth rate is still set by $V$ alone.

### 4.5 Operation counts: what the heap really does

For $V = 1000$ we count, for several graphs, the work done by each algorithm. (a) always reads
exactly $2V^2$ cells. For (b) we count edge scans, heap pushes and stale pops (entries discarded
because a shorter route had already been found). The last column, $V\ln(E/V)$, is only a rough
reference scale for the number of pushes on random graphs.

In [ ]:
n = 1000 if not QUICK_C else 400
rng = random.Random(SEED + 2)
scenarios = [
    ("random, sparse |E|=5|V|", random_graph_edges(n, 5 * n, rng)),
    ("random, half-dense", random_graph_edges(n, n * (n - 1) // 2, rng)),
    ("random, complete", random_graph_edges(n, n * (n - 1), rng)),
    ("adversarial", adversarial_edges(n)),
]

print(f"|V| = {n}.  (a) reads exactly 2|V|^2 = {2 * n * n} cells in every case.\n")
header = (f"{'graph':>24} {'|E|':>9} {'(b) edge scans':>15} {'(b) pushes':>11} "
          f"{'(b) stale pops':>15} {'max heap size':>14} {'|V|ln(|E|/|V|)':>15}")
print(header)
print("-" * len(header))
for name, edges in scenarios:
    _, ops = dijkstraB_counted(to_adj_list(n, edges), 0)
    ref = n * math.log(len(edges) / n)
    print(f"{name:>24} {len(edges):>9} {ops['edge_scans']:>15} {ops['pushes']:>11} "
          f"{ops['stale_pops']:>15} {ops['max_heap']:>14} {ref:>15.0f}")

**Observations**
- **On random graphs the heap is almost idle.** (b) scans every edge once, but pushes only about
  1,500 entries on the sparse graph and about 4,500 on the dense ones, which is under 1% of $E$ in
  the dense case. A vertex's distance is improved only a handful of times before it is final. This is
  why (b) behaves like a plain $\Theta(E)$ scan on random inputs, and why 4.3 can ignore the heap.
- **On the adversarial graph the heap is busy.** Pushes equal $E+1$, nearly all of them become
  stale pops, and the heap grows to hundreds of thousands of entries. This is the
  $\Theta(E\log V)$ worst case, and it is also why (b) is much slower there.
- **(a) is identical in all four cases**: $2V^2 = 2{,}000{,}000$ cell reads.

---
## 5. Discussion: which is better, and when

| Situation | Better | Reason |
|---|---|---|
| **Sparse graph** ($E = O(V)$, or up to roughly 10% of all possible edges) | **(b)** | $\Theta(V\log V)$ against $\Theta(V^2)$. About 65x to 135x faster at $V \ge 1000$ with $E = 5V$, and at 10% density still about 4x to 6x. |
| **Moderately dense**, about 10-40% | **(b)** or a tie | The advantage of (b) shrinks as $E$ grows, and disappears somewhere around 25-70% depending on the machine. |
| **Very dense graph** ($E$ close to $V^2$) | **(a)** | Both are $\Theta(V^2)$ at best, but (a) has the smaller constant (cheaper steps, better memory access) and no heap overhead. On a complete graph (a) was about 1.4x to 3x faster at $V \ge 1000$. |
| **Weights that cause many decrease-keys** (adversarial or structured) | **(a)** | The worst case of (b) is $\Theta(E\log V)$. We measured (b) at 7x to 17x slower than (a). (a) is $\Theta(V^2)$ whatever the weights. |
| **Large $V$ with limited memory** | **(b)** | Needs $V + E$ entries instead of $V^2$ cells. |
| **Need for a predictable running time, or very simple code** | **(a)** | The cost depends only on $V$, and the code is short and has no library dependence. |

**Overall.** (b) is the better general-purpose choice. Most graphs met in practice (road networks,
communication networks, social graphs) are sparse, with $E$ far below $V^2/\log V$, and there (b)
is faster by orders of magnitude. (a) is better when the graph is very dense, when $E$ is
$\Theta(V^2)$ and no algorithm can avoid reading $\Theta(V^2)$ edges, and it is the safer choice
when the running time must not depend on the weights.

**Why the answer is not just "$V^2$ against $(V+E)\log V$".**
1. The worst-case bound of (b), $E\log V$, is rarely reached. With random weights (b) behaved like
   $\Theta(E)$, so it stayed competitive far into the dense range. The theory alone would put the
   crossover at $V^2/\log V$, but in our runs it was a fixed fraction of the maximum (about 25-70%),
   determined by the ratio of the per-step constants.
2. Constants matter in the middle range. In our Python runs an edge scan in (b) cost about 2x to 6x
   more than a cell read in (a), which moved the crossover far from where the asymptotics alone
   would put it. On other hardware or in another language the crossover will move, but its
   existence and the ordering at the two extremes will not.

**Limitations of this study.** Pure Python; $V$ up to 2000; random weights in
$[1, 100]$; every vertex reachable; a single source; one adversarial family. The values quoted
(75x to 135x, the 25-70% crossover, 7x to 17x) are ranges over our runs and will vary between runs and machines.

**Beyond this project.** Replacing the binary heap with a Fibonacci heap gives $O(E + V\log V)$,
which is never worse than (a) asymptotically (since $E + V\log V = O(V^2)$). In practice its large
constants mean that simple binary-heap versions like (b) are usually faster on real inputs.

---
## 6. Conclusion

| | (a) matrix + array | (b) adjacency lists + heap |
|---|---|---|
| Time | $\Theta(V^2)$ in every case | $O((V+E)\log V)$ worst case; about $\Theta(E)$ on random weights |
| Space | $\Theta(V^2)$ | $\Theta(V+E)$ |
| Depends on $E$? | No | Yes, roughly linearly |
| Depends on the weights? | No | Yes (through the number of heap pushes) |
| Wins on | very dense graphs; guaranteed running time | sparse and moderately dense graphs; large $V$ |

The experiments confirm the theory: (a) is flat in $E$ and quadratic in $V$; (b) grows with $E$ and
is far ahead on sparse graphs (roughly 100x for $E = 5V$) but falls behind on very dense ones
(about 1.4x to 3x slower on complete graphs at $V \ge 1000$, and 7x to 17x slower on a graph built to
defeat it). Because most graphs are sparse, **(b) is the better choice in general**, and (a) is
preferable for very dense graphs or when a weight-independent running time is required.